In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.colors import ListedColormap
import seaborn as sns
import ast
import math
from matplotlib.colors import LinearSegmentedColormap
from tqdm.notebook import tqdm
import scienceplots

plt.style.use("default")
plt.style.use(["science", "grid"])

In [2]:
plt.style.use("default")
plt.style.use(["science", "grid"])

In [3]:
root = Path('..')
Images_Path = Path(root / "Images")
Results_Path = Path(root / "Results")

Iteration_Plots_Path = Path(Images_Path / 'Iteration_PyPlots/')
Projection_Plots_Path = Path(Images_Path / 'Projection_PyPlots/')
Control_Plots_Path = Path(Images_Path / 'Control_PyPlots/')
Control_Comparison_Plots_Path = Path(Images_Path / 'Control_Comparison_PyPlots/')
Projection_Comparison_Plots_Path = Path(Images_Path / 'Projection_Comparison_PyPlots/')

High_Difference_Comparison_Plots_Path = Path(Projection_Comparison_Plots_Path / 'High_Difference_Comparison_PyPlots/')
Low_Difference_Comparison_Plots_Path = Path(Projection_Comparison_Plots_Path/ 'Low_Difference_Comparison_PyPlots/')

Empty_Plots_Path = Path(Iteration_Plots_Path / "Empty/")
Full_Plots_Path = Path(Iteration_Plots_Path / "Full/")

Empty_Plots_Path.mkdir(parents=True, exist_ok=True)
Full_Plots_Path.mkdir(exist_ok=True)

Projection_Plots_Path.mkdir(parents=True, exist_ok=True)
Results_Path.mkdir(parents=True, exist_ok=True)

Projection_Comparison_Plots_Path.mkdir(parents=True, exist_ok=True)
High_Difference_Comparison_Plots_Path.mkdir(parents=True, exist_ok=True)
Low_Difference_Comparison_Plots_Path.mkdir(parents=True, exist_ok=True)

Control_Plots_Path.mkdir(parents=True, exist_ok=True)
Control_Comparison_Plots_Path.mkdir(parents=True, exist_ok=True)

In [4]:
data = pd.read_csv(Results_Path /  "results.csv")
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 135000 entries, 0 to 134999
Data columns (total 11 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   alpha               135000 non-null  float64
 1   position_over_time  44959 non-null   object 
 2   final_position      135000 non-null  object 
 3   control_over_time   44959 non-null   object 
 4   method              135000 non-null  object 
 5   N                   135000 non-null  int64  
 6   x0                  135000 non-null  object 
 7   x_d                 135000 non-null  object 
 8   mass                135000 non-null  float64
 9   k_spring            135000 non-null  float64
 10  iterations          135000 non-null  int64  
dtypes: float64(3), int64(2), object(6)
memory usage: 11.3+ MB


In [5]:
data['x_d'].apply(tuple)

def str_to_arr(x : str) -> np.array:
  return np.fromstring(x[1:-1], sep=',')

def str_to_list(x : str) -> list:
  return str_to_arr(x).tolist()

def str_to_tuple(x : str) -> tuple:
  return tuple(str_to_arr(x))

array_str_eval = lambda x: np.nan if type(x) != str and math.isnan(x) else np.array(ast.literal_eval(x))

data['final_position'] = data['final_position'].apply(str_to_tuple)
data['x0'] = data['x0'].apply(str_to_tuple)
data['x_d'] = data['x_d'].apply(str_to_tuple)

data['position_over_time'] = data['position_over_time'].apply( array_str_eval )
data['control_over_time'] = data['control_over_time'].apply( array_str_eval )

data.info()
data.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 135000 entries, 0 to 134999
Data columns (total 11 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   alpha               135000 non-null  float64
 1   position_over_time  44959 non-null   object 
 2   final_position      135000 non-null  object 
 3   control_over_time   44959 non-null   object 
 4   method              135000 non-null  object 
 5   N                   135000 non-null  int64  
 6   x0                  135000 non-null  object 
 7   x_d                 135000 non-null  object 
 8   mass                135000 non-null  float64
 9   k_spring            135000 non-null  float64
 10  iterations          135000 non-null  int64  
dtypes: float64(3), int64(2), object(6)
memory usage: 11.3+ MB


,alpha,position_over_time,final_position,control_over_time,method,N,x0,x_d,mass,k_spring,iterations
0,10.0,"[[1.1111111111111112, 25.555405405405406], [1....","(0.3061078490191818, 21.01667950899807)","[-0.8688655855306409, -0.7600155309564338, -0....",Modified SE1,100,"(1.1111111111111112, 25.555405405405406)","(0.0, 4.0)",1.0,1.00,6
1,10.0,NaN,"(nan, nan)",NaN,MidPoint,100,"(2.0, 1.3121621621621622)","(0.0, 8.0)",1.0,0.50,-1
2,10.0,"[[-2.0, 28.787837837837838], [-2.0, 28.7878378...","(-0.9181296023187153, 20.903036115433377)","[1.0940798803701188, 0.9962020423686996, 0.898...",SE2,200,"(-2.0, 28.787837837837838)","(0.0, 6.0)",1.0,0.50,6
3,10.0,"[[0.6666666666666665, 20.706756756756754], [0....","(0.42465602129139196, 9.686934899972583)","[-0.5749439623855019, -0.5558073553436716, -0....",Modified MidPoint,200,"(0.6666666666666665, 20.706756756756754)","(0.0, 2.0)",1.0,0.25,7
4,10.0,NaN,"(nan, nan)",NaN,SE2,200,"(-1.5555555555555554, 23.13108108108108)","(0.0, 6.0)",1.0,0.50,-1


In [6]:
data_cp = data[data['x0'].str[1] <= 5]
converged_entries = data_cp[data_cp['position_over_time'].notna()]

In [7]:
def make_control_plot(control : np.ndarray, method : str, m : float, k : float, N : int, alpha : float, x0_str : str, x_d_str : str):
  label = (
    f"Method : {method}\n"
    f"m = {m}\n"
    f"k = {k}\n"
    f"N = {N}\n"
    f"alpha = {alpha}\n"
    rf"$\boldsymbol{{x}}_0 = \boldsymbol{{\left({x0_str}\right)}}$" "\n"
    rf"$\boldsymbol{{x}}_d = \boldsymbol{{\left({x_d_str}\right)}}$"
  )

  x_range = np.linspace(0, 20, N+1)[0:len(control)]

  fig = plt.figure(figsize=(10, 6))
  plt.plot(x_range, control, linestyle='dashed', label="Control", marker='o', lw=2, color="red", ms=5, mfc='blue', mec='black')
  plt.xlabel("Time")
  plt.ylabel("Control")
  plt.title(f"Control over time\n{method}", fontsize="large")
  plt.gca().text(
    0.98, 0.98,
    label,
    fontsize=10,
    ha='right',
    va='top',
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
    transform=plt.gca().transAxes
  )

  plt.close(fig)
  return fig

In [8]:
groups = data_cp.groupby(by=["method", "k_spring", "x_d", "alpha", "N", "mass"])
iter_color = LinearSegmentedColormap.from_list(
    "iter_smooth",
    ["#1E9600", "#FFF200", "#FF0000"]
)

for (method, k_spring, x_d, alpha, N, m) ,g in tqdm(
  groups,
  total=len(groups),
  desc="Generating iteration plots"
):
    fig = plt.figure(figsize=(10, 6))

    plt.hlines(y = 0.0, xmin=-2, xmax=2, linestyles='dashed')
    plt.vlines(x = 0.0, ymin = 0, ymax = max(x_d[1], 2.0), linestyles='dashed')
    plt.scatter(x_d[0], x_d[1], marker='*', c='purple', label=f"Desired Position [{x_d[0]}, {x_d[1]}]", sizes=np.array([60]))

    x_d_str = ", ".join(f"{v:.2f}" for v in x_d)
    X, Y, C = [], [], []
    for index, row in g.iterrows():
      x0 = row.loc['x0']

      x0_str = ", ".join(f"{v:.2f}" for v in x0)

      final_position = row.loc['final_position']
      iteration = row.loc['iterations']
      control = row.loc['control_over_time']

      if iteration == -1:
        plt.scatter(x0[0], x0[1], c='k')
      else:
        x, y = x0[0], x0[1]
        X.append(x0[0])
        Y.append(x0[1])
        C.append(iteration)

        dx, dy = final_position[0] - x, final_position[1] - y
        plt.arrow(x, y, dx, dy, width=0.025)

      if isinstance(control, np.ndarray):
        control_plot_fig = make_control_plot(control, method, m, k_spring, N, alpha, x0_str = x0_str, x_d_str = x_d_str)
        control_plot_fig.savefig(Control_Plots_Path / f"ControlPlot_{method}_k={k_spring}_h={x_d[1]}_N={N}_m={m}_x0={x0_str}.pdf", dpi=300, format="pdf")

    sc = plt.scatter(X, Y, c=C, cmap = iter_color, vmin = 0, vmax = 30, sizes=np.full(len(X), 80))
    plt.colorbar(sc, label="Iterations")
    plt.title(f"Iteration Plot for {method}", fontsize="large")

    legend = plt.legend(fancybox=False, edgecolor="black")
    legend.get_frame().set_linewidth(0.5)

    plt.xlim(-2.2, 2.2)
    plt.ylim(-0.2, max(5.2, x_d[1] + 0.2))

    ax = plt.gca()
    ax.text(
    0.02, 0.98,
    f"Method : {method}\nm = {m}\nk = {k_spring}\nN = {N}\nalpha = {alpha}",
    fontsize=10,
    ha='left',
    va='top',
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
    transform=ax.transAxes
  )

    plt.savefig((Empty_Plots_Path if len(X) == 0 else Full_Plots_Path)  / f"IterationPlot_{method}_k={k_spring}_h={x_d[1]}_N={N}_m={m}.pdf", dpi=300, format="pdf")
    plt.close()





Generating iteration plots:   0%|          | 0/180 [00:00<?, ?it/s]

In [9]:
converged_rows = converged_entries.itertuples(index=False)
for row in tqdm(
        converged_rows,
        total=len(converged_entries),
        desc="Generating Trajectory Plots"
):
  position_over_time = np.array(row.position_over_time)
  N = row.N
  k = row.k_spring
  m = row.mass
  method = row.method
  x_d = row.x_d
  x0 = row.x0

  x_d_str = ", ".join(f"{v:.2f}" for v in x_d)
  x0_str = ", ".join(f"{v:.2f}" for v in x0)

  x = position_over_time[:, 0]
  y = position_over_time[:, 1]
  dx = np.diff(x)
  dy = np.diff(y)

  allX = np.hstack((x, x_d[0], [0]))
  allY = np.hstack((y, x_d[1], [0]))

  minX, maxX = allX.min(), allX.max()
  minY, maxY = allY.min(), allY.max()

  speed = np.sqrt(dx**2 + dy**2)

  plt.figure(figsize=(10, 6))
  plt.hlines(y = 0.0, xmin=minX, xmax=maxX, linestyles='dashed')
  plt.vlines(x = 0.0, ymin=minY, ymax=maxY, linestyles='dashed')

  Q = plt.quiver(x[:-1],y[:-1], dx, dy, speed, angles='xy', scale_units='xy', scale=1, cmap='viridis')
  plt.colorbar(mappable=Q, label=r"Step magnitude $| \Delta x \|$")

  plt.scatter([x[0]], [y[0]], marker='o', c='lime', label=f"Starting Position [{x0_str}]")
  plt.scatter(x_d[0], x_d[1], marker='*', c='purple', label=f"Desired Position[{x_d_str}]")


  ax = plt.gca()
  ax.text(
    0.02, 0.98,
    f"Method : {row.method}\nm = {row.mass}\nk = {row.k_spring}\nN = {row.N}\nalpha = {row.alpha}",
    fontsize=10,
    ha='left',
    va='top',
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
    transform=ax.transAxes
  )

  plt.legend()
  plt.xlabel("X")
  plt.ylabel("Y")

  plt.savefig(Projection_Plots_Path / f"ProjectionPlot_{method}_k={k}_h={x_d[1]}_N={N}_m={m}_x0={x0_str}.pdf", dpi=300, format="pdf")
  plt.close()



Generating Trajectory Plots:   0%|          | 0/3755 [00:00<?, ?it/s]

In [10]:
def add_axis_details(ax, minX, maxX, minY, maxY, x0, x_d, m, k, N, alpha, x0_str, x_d_str, method):
  ax.hlines(y = 0.0, xmin=minX, xmax=maxX, linestyles='dashed')
  ax.vlines(x = 0.0, ymin=minY, ymax=maxY, linestyles='dashed')
  ax.set_xlabel("X")
  ax.set_ylabel("Y")
  ax.scatter(x0[0], x0[1], marker='o', c='lime', label=f"Starting Position[{x0_str}]")
  ax.scatter(x_d[0], x_d[1], marker='*', c='purple', label=f"Desired Position[{x_d_str}]")
  ax.text(
  0.02, 0.98,
  f"Method : {method}\nm = {m}\nk = {k}\nN = {N}\nalpha = {alpha}",
  fontsize=10,
  ha='left',
  va='top',
  bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
  transform=ax.transAxes
  )
  ax.legend()

def plt_comparison(alpha, N, k, x0, x_d, m, modified_method_series, non_modified_method_series, method):
  fig, axs = plt.subplots(nrows=2, ncols=1, figsize=(10, 12.5), sharex=True, sharey=True)

  x_d_str = ", ".join(f"{v:.2f}" for v in x_d)
  x0_str = ", ".join(f"{v:.2f}" for v in x0)

  position_over_time = non_modified_method_series['position_over_time'].iloc[0]
  modified_position_over_time = modified_method_series['position_over_time'].iloc[0]

  total_abs_diff = np.abs(position_over_time - modified_position_over_time).sum()

  x = position_over_time[:, 0]
  y = position_over_time[:, 1]
  dx = np.diff(x)
  dy = np.diff(y)
  speed = np.sqrt(dx**2 + dy**2)

  x_modified = modified_position_over_time[:, 0]
  y_modified = modified_position_over_time[:, 1]
  dx_modified = np.diff(x_modified)
  dy_modified = np.diff(y_modified)
  speed_modified = np.sqrt(dx_modified**2 + dy_modified**2)

  allX = np.hstack((x, x_modified, x_d[0], [0]))
  allY = np.hstack((y, y_modified, x_d[1], [0]))

  minX, maxX = allX.min(), allX.max()
  minY, maxY = allY.min(), allY.max()

  add_axis_details(axs[0], minX = minX, minY= minY, maxX = maxX,maxY =  maxY,x0 =  x0,x_d =  x_d,m = m, k = k, N = N,alpha = alpha, x0_str = x0_str, x_d_str = x_d_str, method = f"Modified {method}")
  add_axis_details(axs[1], minX = minX, minY= minY, maxX = maxX,maxY =  maxY,x0 =  x0,x_d =  x_d,m = m, k = k, N = N,alpha = alpha, x0_str = x0_str, x_d_str = x_d_str, method = method)

  Q_modified = axs[0].quiver(
    x_modified[:-1], y_modified[:-1],
    dx_modified, dy_modified,
    speed_modified,
    angles='xy',
    scale_units='xy',
    scale=1,
    cmap='viridis',
  )

  Q = axs[1].quiver(
    x[:-1], y[:-1],
    dx, dy,
    speed,
    angles='xy',
    scale_units='xy',
    scale=1,
    cmap='viridis',
  )


  plt.colorbar(mappable=Q_modified, ax=axs[0], label=r"Step magnitude $\| \Delta x \|$")
  plt.colorbar(mappable=Q, ax=axs[1], label=r"Step magnitude $\| \Delta x \|$")

  plt.close(fig)
  return fig, total_abs_diff



In [11]:
SE1_convergence = converged_entries[converged_entries["method"].str.contains("SE1")]
SE1_Groups = SE1_convergence.groupby(by=['alpha', 'N', 'k_spring', 'x0', 'x_d', 'mass'])
for group_key, group_df in tqdm(
  SE1_Groups,
  total=len(SE1_Groups),
  desc="Generating Comparison of projections plots"
):
  modified_method_series = group_df[group_df["method"].str.contains("Modified")]
  non_modified_method_series = group_df[~group_df["method"].str.contains("Modified")]

  if modified_method_series.empty or non_modified_method_series.empty:
    continue

  alpha, N, k, x0, x_d, m = group_key
  x0_str = ", ".join(f"{v:.2f}" for v in x0)

  fig, total_abs_diff = plt_comparison(alpha, N, k, x0, x_d, m, modified_method_series, non_modified_method_series, "SE1")
  filename = f"Projection_Comparison_SE1_k={k}_h={x_d[1]}_N={N}_m={m}_x0={x0_str}.pdf"
  if total_abs_diff <= 10:
    fig.savefig(Low_Difference_Comparison_Plots_Path / filename, bbox_inches='tight', transparent=True)
  else:
    fig.savefig(High_Difference_Comparison_Plots_Path / filename, bbox_inches='tight', transparent=True)



Generating Comparison of projections plots:   0%|          | 0/928 [00:00<?, ?it/s]

In [12]:
SE2_convergence = converged_entries[converged_entries["method"].str.contains("SE2")]
SE2_Groups = SE2_convergence.groupby(by=['alpha', 'N', 'k_spring', 'x0', 'x_d', 'mass'])

for group_key, group_df in tqdm(
  SE2_Groups,
  total=len(SE2_Groups),
  desc="Generating Comparison of projections plots"
):
  modified_method_series = group_df[group_df["method"].str.contains("Modified")]
  non_modified_method_series = group_df[~group_df["method"].str.contains("Modified")]

  if modified_method_series.empty or non_modified_method_series.empty:
    continue

  alpha, N, k, x0, x_d, m = group_key
  x0_str = ", ".join(f"{v:.2f}" for v in x0)

  fig, total_abs_diff = plt_comparison(alpha, N, k, x0, x_d, m, modified_method_series, non_modified_method_series, "SE2")
  filename = f"Projection_Comparison_SE2_k={k}_h={x_d[1]}_N={N}_m={m}_x0={x0_str}.pdf"
  if total_abs_diff <= 10:
    fig.savefig(Low_Difference_Comparison_Plots_Path / filename, bbox_inches='tight', transparent=True)
  else:
    fig.savefig(High_Difference_Comparison_Plots_Path / filename, bbox_inches='tight', transparent=True)


Generating Comparison of projections plots:   0%|          | 0/468 [00:00<?, ?it/s]

In [13]:
MidPoint_convergence = converged_entries[converged_entries["method"].str.contains("MidPoint")]
MidPoint_Groups = MidPoint_convergence.groupby(by=['alpha', 'N', 'k_spring', 'x0', 'x_d', 'mass'])

for group_key, group_df in tqdm(
  MidPoint_Groups,
  total=len(MidPoint_Groups),
  desc="Generating Comparison of projections plots"
):
  modified_method_series = group_df[group_df["method"].str.contains("Modified")]
  non_modified_method_series = group_df[~group_df["method"].str.contains("Modified")]

  if modified_method_series.empty or non_modified_method_series.empty:
    continue

  alpha, N, k, x0, x_d, m = group_key
  x0_str = ", ".join(f"{v:.2f}" for v in x0)

  fig, total_abs_diff = plt_comparison(alpha, N, k, x0, x_d, m, modified_method_series, non_modified_method_series, "MidPoint")
  filename = f"Projection_Comparison_MidPoint_k={k}_h={x_d[1]}_N={N}_m={m}_x0={x0_str}.pdf"
  if total_abs_diff <= 10:
    fig.savefig(Low_Difference_Comparison_Plots_Path / filename, bbox_inches='tight', transparent=True)
  else:
    fig.savefig(High_Difference_Comparison_Plots_Path / filename, bbox_inches='tight', transparent=True)

Generating Comparison of projections plots:   0%|          | 0/709 [00:00<?, ?it/s]